# MovieLens Recommender — Stage 1: Data + Honest Baseline

Goal of the whole project: a movie recommender with **two-stage retrieval + ranking**,
evaluated honestly. Stage 1 builds the foundation everything else is judged against:

1. Load MovieLens-25M (25 million ratings).
2. Split **by time** (train on older ratings, test on newer) — not randomly. Random
   splitting lets the model "see the future," inflating scores dishonestly.
3. Build a **popularity baseline**: recommend the globally most-popular movies to
   everyone. Simple, but it's the yardstick every learned model must beat.
4. Measure it with **Recall@K** and **NDCG@K** (the standard recsys metrics).

If a fancy model can't beat "just recommend popular stuff," it isn't earning its keep.
Establishing that bar first is the honest way to build a recommender.

**Runtime:** GPU is not needed for Stage 1 (this is pandas/CPU work).

In [ ]:
import pandas as pd, numpy as np
print("pandas", pd.__version__)

## 1. Get the data
MovieLens-25M is a standard public dataset from GroupLens — no login/token needed.

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-25m.zip
!unzip -q -o ml-25m.zip
!ls ml-25m

## 2. Load ratings
Each row: a user rated a movie, at a timestamp. That timestamp lets us split by time.

In [ ]:
ratings = pd.read_csv('ml-25m/ratings.csv')  # userId, movieId, rating, timestamp
movies  = pd.read_csv('ml-25m/movies.csv')   # movieId, title, genres
print(ratings.shape, "ratings")
print("users:", ratings.userId.nunique(), "| movies:", ratings.movieId.nunique())
ratings.head()

## 3. Treat as implicit feedback
Keep ratings >= 4.0 as a positive signal ("this user liked this movie"); drop the rest.
This is the standard implicit-feedback setup real recommenders use.

In [ ]:
pos = ratings[ratings.rating >= 4.0].copy()
print(f"kept {len(pos):,} positive interactions ({len(pos)/len(ratings)*100:.1f}%)")

# Keep reasonably active users and popular-enough movies (removes ultra-sparse noise).
uc = pos.userId.value_counts(); ic = pos.movieId.value_counts()
pos = pos[pos.userId.isin(uc[uc>=5].index) & pos.movieId.isin(ic[ic>=5].index)]
print(f"after filtering: {len(pos):,} interactions, "
      f"{pos.userId.nunique():,} users, {pos.movieId.nunique():,} movies")

## 4. The temporal split (the important part)
Hold out each user's **most recent** ~20% of liked movies as test; earlier ones are
train. This mimics reality: predict what a user likes *next* from their *past*. A
random split would leak future interactions into training and lie about performance.

In [ ]:
pos = pos.sort_values(['userId','timestamp'])

def split_user(g):
    n_test = max(1, int(len(g)*0.2))
    g = g.copy(); g['is_test'] = False
    g.iloc[-n_test:, g.columns.get_loc('is_test')] = True
    return g

pos = pos.groupby('userId', group_keys=False).apply(split_user)
train = pos[~pos.is_test]; test = pos[pos.is_test]
print(f"train: {len(train):,}  |  test: {len(test):,}")

## 5. The popularity baseline
Rank movies by training-set popularity; recommend that same top-K to **every** user
(skipping what they already liked in training). No personalization — but a tough bar.

In [ ]:
K = 10
popularity = train.movieId.value_counts()
top_popular = popularity.index.tolist()
train_seen = train.groupby('userId').movieId.apply(set).to_dict()

def recommend_popular(user_id, k=K):
    seen = train_seen.get(user_id, set()); recs = []
    for m in top_popular:
        if m not in seen:
            recs.append(m)
            if len(recs)==k: break
    return recs

## 6. Evaluate with Recall@K and NDCG@K
- **Recall@K**: of the movies the user actually liked in test, how many made our top-K?
- **NDCG@K**: like Recall but rewards ranking correct items higher.
Averaged over a sample of users for speed.

In [ ]:
def dcg(hits): return sum(1.0/np.log2(i+2) for i,h in enumerate(hits) if h)

def evaluate(recommend_fn, k=K, n_users=3000, seed=0):
    test_by_user = test.groupby('userId').movieId.apply(set).to_dict()
    users = list(test_by_user.keys())
    sample = np.random.default_rng(seed).choice(users, size=min(n_users,len(users)), replace=False)
    recalls, ndcgs = [], []
    for u in sample:
        truth = test_by_user[u]
        if not truth: continue
        recs = recommend_fn(u,k)
        hits = [1 if m in truth else 0 for m in recs]
        recalls.append(sum(hits)/min(len(truth),k))
        ideal = dcg([1]*min(len(truth),k))
        ndcgs.append(dcg(hits)/ideal if ideal>0 else 0.0)
    return float(np.mean(recalls)), float(np.mean(ndcgs))

rec, ndcg_score = evaluate(recommend_popular)
print(f"POPULARITY BASELINE  ->  Recall@{K}: {rec:.4f}   NDCG@{K}: {ndcg_score:.4f}")

## 7. What we established
That Recall@K / NDCG@K is **the number to beat.** Every model from here (matrix
factorization → two-tower → ranker) must beat this simple baseline to justify its
complexity. Keeping this honest yardstick front-and-center is the core of the project.

**Next (Stage 2):** matrix factorization — learn an embedding per user and per movie,
so recommendations become personalized instead of one-size-fits-all.